In [ ]:
preds = jnp.load(
    "benchmarks/disease_mapping/results/MCMC_2025-04-12_14-01-48/gp_pointwise/predictions.npz"
)

In [ ]:
dict(preds)

In [ ]:
s = preds["s"]
theta = preds["theta"]

mean = theta.mean(0)
s.shape, mean.shape

In [ ]:
lon, lat = s.T
res = 150 / 3600
lon_range = np.arange(lon.min() - 0.5 * res, lon.max() + res, res)
lat_range = np.arange(lat.min() - 0.5 * res, lat.max() + res, res)

lon_range.shape, lat_range.shape


In [ ]:
M = len(lon_range) - 1
N = len(lat_range) - 1
print(M, N)

In [ ]:
X, Y = np.meshgrid(lon_range, lat_range)

binned_lon = np.digitize(lon, lon_range) - 1
binned_lat = np.digitize(lat, lat_range) - 1
data = np.ones((N, M)) * np.nan

for i, j, m in zip(binned_lon, binned_lat, mean):
    data[j, i] = m
print(data.shape, X.shape, Y.shape)
plt.pcolormesh(X, Y, data)
plt.axis("equal")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection


lons, lats = s.T


# shape = get_shape("MOZ")
# plot_polygon(shape, ax=ax, add_points=False)
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_aspect("equal")

res = 150 / 3600
cm = plt.get_cmap("viridis")
norm = plt.Normalize(vmin=0, vmax=1)
colors = cm(norm(mean), alpha=1)
sm = plt.cm.ScalarMappable(norm, cm)

anchor_points = s - res / 2
circles = PatchCollection(
    [plt.Rectangle(s, res, res, color=c, linewidth=0) for s, c in zip(s, colors)],
    match_original=True,
)
ax.add_collection(circles, autolim=True)
ax.set_xlim(s[:, 0].min() - 1, s[:, 0].max() + 1)
ax.set_ylim(s[:, 1].min() - 1, s[:, 1].max() + 1)
fig.colorbar(sm, ax=ax)
fig.show()


In [ ]:
from benchmarks.disease_mapping.visualize import plot_predictions

data = jnp.load("benchmarks/disease_mapping/results/MCMC_2025-04-12_14-01-48/data.npz")
plot_predictions(s, theta, data=data)

In [ ]:
import plotly.express as px

px.scatter_map(
    lon=s[:, 0],
    lat=s[:, 1],
    color=mean,
    color_continuous_scale="viridis",
    range_color=[0, 1],
)

In [ ]:
fig = px.scatter_map(
    lon=data["s"][:, 0],
    lat=data["s"][:, 1],
    color=data["n_pos"] / data["n"],
    color_continuous_scale="viridis",
    range_color=[0, 1],
    size=data["n"],
)
fig.update_geos(fitbounds="locations")
fig.show()

with open("image.png", "wb") as f:
    f.write(fig.to_image("png"))

In [ ]:
s, theta